In [1]:
import pandas as pd
import numpy as np
from io import StringIO

# annot_df = pd.read_csv('../gen_datasets/datasets/csa_annot.csv', sep='\t')
# annot_df['sequence_len'] = annot_df['Sequence'].apply(lambda x: len(x) if isinstance(x, str) else 0)

import os
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

def hits_to_fasta(hit_df, out_dir, whitelist=None):
    
    # hit_df['win_size'] = hit_df.apply(lambda x: x['tlen'] / x['qlen'] < 1.75, axis=1)
    # hit_df = hit_df[hit_df['win_size']]

    def to_fasta(f, pid_l, seq_l):
        seq_records = [
            SeqRecord(Seq(seq), id=str(pid), description="")
            for pid, seq in zip(pid_l, seq_l)
        ]
        SeqIO.write(seq_records, f, "fasta")

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    for group in hit_df.groupby('query'):
        query_id = group[0]
        if whitelist is None or query_id in whitelist:
            pass
            # print(f'Processing {query_id}')
            # print(query_id, group[1])
        else:
            continue
        # Filter for sequences with length within 2 standard deviations of the mean
        group_df = group[1]
        init_len = len(group_df)
        group_df = group_df[group_df['pident'] <= 95]
        m, std = np.mean(group_df['tlen']), np.std(group_df['tlen']) + 1
        group_df = group_df[(group_df['tlen'] > m - 2 * std) & (group_df['tlen'] < m + 2 * std) & (group_df['tlen'] < 1500)]

        if(len(group_df) < 5):
            # print(group_df, query_id, init_len, m, std)
            continue

        group_id = list(group_df['subject'])
        group_seq = list(group_df['tseq'])
        
        if(len(group_id) > 450):
            group_id = group_id[:450]
            group_seq = group_seq[:450]

        # print(group_df['qseq'].values[0])
        if(len(group_df['qseq'].values[0]) > 850):
            print(f'Skipping {query_id} due to long query sequence length: {len(group_df["qseq"].values[0])}')
            print(len(group_df['qseq'].values[0]))
            continue


        if(not any(query_id in s for s in group_id)):
            group_id.append(query_id)
            group_seq.append(group_df['qseq'].values[0])
        
        print('max len', max([len(seq) for seq in group_seq]), query_id, len(group_seq))
        to_fasta(f'{out_dir}/{query_id}_homologues.fasta', group_id, group_seq)

In [2]:
hit_df = pd.read_csv('ip_domain_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
# group_sizes = hit_df.groupby('query').size()
# whitelist = ['O76290', 'P13448:P13449', 'P56740', 'Q93EK7', 'Q967M2']
hits_to_fasta(hit_df, 'uniref_msa/ip_domain_msa')

max len 242 ipdom0 299
max len 454 ipdom1 312
max len 685 ipdom10 289
max len 332 ipdom100 315
max len 800 ipdom101 370
max len 792 ipdom102 368
max len 154 ipdom103 360
max len 338 ipdom104 342
max len 136 ipdom105 414
max len 647 ipdom106 328
max len 644 ipdom107 323
max len 647 ipdom108 328
max len 689 ipdom109 316
max len 1035 ipdom11 425
max len 733 ipdom110 300
max len 273 ipdom111 297
max len 212 ipdom112 338
max len 634 ipdom113 314
max len 771 ipdom114 300
max len 690 ipdom115 342
max len 732 ipdom116 320
max len 492 ipdom117 283
max len 733 ipdom118 331
max len 729 ipdom119 286
max len 634 ipdom12 335
max len 924 ipdom120 347
max len 924 ipdom121 354
max len 394 ipdom122 362
max len 686 ipdom123 331
max len 138 ipdom124 317
max len 527 ipdom125 292
max len 748 ipdom126 292
max len 719 ipdom127 305
max len 725 ipdom128 325
max len 764 ipdom129 293
max len 692 ipdom13 311
max len 441 ipdom130 333
max len 387 ipdom131 309
max len 728 ipdom132 304
max len 740 ipdom133 350
max len

In [41]:
hit_df = pd.read_csv('llps_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/llps_msa')

hit_df = pd.read_csv('elms_aln.m8', sep='\t', header=None, 
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen', 
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/elms_msa')

Skipping A0QSY1 due to long query sequence length: 899
899
Skipping A1Z813 due to long query sequence length: 1184
1184
max len 880 D0PV95 290
max len 904 D8V196 294
Skipping F1R237 due to long query sequence length: 2117
2117
max len 893 G5EBV6 33
max len 999 H0WFA5 308
Skipping J3QQ18 due to long query sequence length: 1308
1308
max len 658 O00401 293
max len 760 O00571 277
max len 1062 O13828 327
max len 355 O43561 173
max len 771 O43781 301
max len 959 O43791 256
Skipping O60500 due to long query sequence length: 1241
1241
max len 876 O60563 305
Skipping O60885 due to long query sequence length: 1362
1362
Skipping O62011 due to long query sequence length: 1054
1054
Skipping O65934 due to long query sequence length: 865
865
max len 872 O74718 315
max len 917 O94752 438
max len 421 P00873 350
max len 740 P03372 304
max len 327 P03520 30
max len 622 P03521 306
Skipping P04050 due to long query sequence length: 1733
1733
max len 804 P04147 299
max len 592 P04156 268
max len 817 P05067 

In [ ]:
# csa uses UniprotID as query IDs (from gen_datasets/datasets/csa.fasta)
hit_df = pd.read_csv('csa_aln.m8', sep='\t', header=None,
                     names=['query', 'subject', 'fident', 'pident', 'nident', 'qlen', 'tlen',
                            'qstart', 'qend', 'tstart', 'tend', 'evalue', 'bits', 'qseq', 'tseq'])
hits_to_fasta(hit_df, 'uniref_msa/csa_msa')